In [2]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359

maxval=1e9
minval=1e-9



2025-06-23 14:51:57.226057: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-23 14:51:57.226126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-23 14:51:57.227019: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-23 14:51:57.233297: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-23 14:51:58.213942: W tensorflow/compiler/tf2

In [3]:
# os.chdir('SmartPix/data_generator')
os.chdir('/home/das214/SmartPix/mlp_enc_dev')
!pwd

/home/das214/SmartPix/mlp_enc_dev


In [4]:
from DG.OptimizedDataGenerator_v2 import OptimizedDataGenerator
from losses.diag_loss_nll import custom_diag_loss
from models.mlp_encoder_model import CreateModel

In [5]:
dataset_base_dir = "/depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train")
dataset_test_dir = os.path.join(dataset_base_dir, "test")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val")

batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_test_dir))

In [6]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_test_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,

#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )

# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# # training generator
# start_time = time.time()
# training_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_train_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = batch_size,
#     # optimize_batch_size = True,
#     file_count = train_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, # True 

#     tfrecords_dir = tfrecords_dir_train,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Training generator %s seconds ---" % (time.time() - start_time))

In [7]:
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=True,
)


Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json


In [20]:
diag_model=CreateModel(shape = (16,16,2), output = 8)
diag_model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_diag_loss,
    run_eagerly=True
)

diag_model.summary()

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls/ (InputLayer)    [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 q_activation_6 (QActivatio  (None, 16, 16, 2)            0         ['input_pxls/[0][0]']         
 n)                                                                                               
                                                                                                  
 q_activation_7 (QActivatio  (None, 16, 16, 2)            0         ['input_pxls/[0][0]']         
 n)                                                                                               
                                                                                 

In [21]:
from datetime import datetime

fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [22]:
print(fingerprint)

780b36a7


In [23]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback

early_stopping_patience = 50

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort()
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [24]:
history = diag_model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=1000,
        shuffle=False,
        verbose=1
    )

Epoch 1/1000


84/84 [==============================] - ETA: 0s - loss: nan
Epoch 1: val_loss did not improve from inf
84/84 [==============================] - 12s 126ms/step - loss: nan - val_loss: nan
Epoch 2/1000
84/84 [==============================] - ETA: 0s - loss: nan
Epoch 2: val_loss did not improve from inf
84/84 [==============================] - 11s 125ms/step - loss: nan - val_loss: nan
Epoch 3/1000
84/84 [==============================] - ETA: 0s - loss: nan
Epoch 3: val_loss did not improve from inf
84/84 [==============================] - 11s 125ms/step - loss: nan - val_loss: nan
Epoch 4/1000
84/84 [==============================] - ETA: 0s - loss: nan
Epoch 4: val_loss did not improve from inf
84/84 [==============================] - 11s 126ms/step - loss: nan - val_loss: nan
Epoch 5/1000
84/84 [==============================] - ETA: 0s - loss: nan
Epoch 5: val_loss did not improve from inf
84/84 [==============================] - 11s 126ms/step - loss: nan - val_loss: nan
Epoch 6/

KeyboardInterrupt: 

In [ ]:
X_batch, y_batch = training_generator[0]

print("--- Running a single forward pass... ---")
try:
    # Get the model's raw predictions
    predictions = diag_model(X_batch, training=True)

    # Use TensorFlow's built-in checker
    tf.debugging.check_numerics(predictions, "Model predictions contain NaN or Inf!")

    print("✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).")
    print("\nSample of predictions (first 5):")
    print(predictions.numpy()[:5])

except Exception as e:
    print(f"❌ FAILURE: The NaN is being generated inside the model's forward pass.")
    print(f"Error: {e}")


--- Checking a Batch of Data ---
Shape of X: (5000, 16, 16, 2)
Shape of y: (5000, 4)

Are there any NaNs in X_batch? -> False
Are there any NaNs in y_batch? -> False

Stats for X_batch: Min=-1.0000, Max=0.8750, Mean=0.0117
Stats for y_batch: Min=-1.0634, Max=1.0549, Mean=-0.0081
